# CNN Feature Map Analysis for Sleep-Stage Classification

Visualizing what the 1D CNN learns from EEG waveforms across different sleep stages.

**Goal:** Understand which temporal patterns drive the model's decisions by examining intermediate activations and learned filters.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import signal
import sys, os

sys.path.insert(0, '.')
from data_loader import load_data, STAGE_NAMES
from preprocess import normalize_epoch
from model import SleepStageCNN

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

STAGE_COLORS = ['#3366cc', '#dc3912', '#ff9900', '#109618', '#990099']
OUTPUT_DIR = 'figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
X_raw, y, sfreq, _ = load_data()
X = np.array([normalize_epoch(e) for e in X_raw], dtype=np.float32)
print(f"Dataset: {X.shape}")
print(f"Raw range: [{X_raw.min():.2e}, {X_raw.max():.2e}]  Norm range: [{X.min():.2f}, {X.max():.2f}]")

device = torch.device('cpu')
model = SleepStageCNN(3000, 5).to(device)
model.load_state_dict(torch.load('sleep_stage_cnn.pth', map_location=device, weights_only=True))
model.eval()
print('Model loaded.')

---
## 1. Extract Intermediate Activations

We register forward hooks on each convolutional layer to capture the output feature maps when passing example epochs through the network.

In [ ]:
activations = {}

def make_hook(name):
    def hook(module, inp, out):
        activations[name] = out.detach()
    return hook

hooks = []
for name, mod in model.named_modules():
    if isinstance(mod, torch.nn.Conv1d):
        h = mod.register_forward_hook(make_hook(name))
        hooks.append(h)

# Pick one example epoch per stage
stage_examples = {}
for i, s in enumerate(STAGE_NAMES):
    idx = np.where(y == i)[0][0]
    stage_examples[s] = X[idx]

stage_outputs = {}
for stage_name, epoch in stage_examples.items():
    t = torch.tensor(epoch, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    _ = model(t)
    stage_outputs[stage_name] = {k: v.clone() for k, v in activations.items()}

for h in hooks:
    h.remove()

print('Activations captured for layers:', list(activations.keys()))
for s in STAGE_NAMES:
    print(f'\n{s}:')
    for layer, act in stage_outputs[s].items():
        print(f'  {layer}: {act.shape}')

---
## 2. Visualize Learned Conv1 Filters (Kernel Weights)

The first convolutional layer learns 16 filters of length 7 samples (70 ms at 100 Hz). These are the primitive waveform patterns the model detects in the raw EEG signal.

In [ ]:
conv1_weights = model.conv1.weight.data.cpu().numpy()  # (16, 1, 7)

fig, axes = plt.subplots(4, 4, figsize=(12, 8))
fig.suptitle('Conv1 Learned Filters (16 filters, kernel size=7 = 70ms)', fontsize=13)
t_kernel = np.arange(7) / sfreq * 1000  # ms

for i in range(16):
    ax = axes[i // 4, i % 4]
    ax.plot(t_kernel, conv1_weights[i, 0], color='black', linewidth=1.2)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_title(f'Filter {i+1}', fontsize=9)
    ax.set_ylim(conv1_weights.min() - 0.1, conv1_weights.max() + 0.1)

for ax in axes.flat:
    ax.set_xlabel('Time (ms)', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'conv1_filters.png'), dpi=150, bbox_inches='tight')
plt.show()

### Filter Interpretation

Each filter acts as a matched template — it slides across the EEG signal and activates strongly where the local waveform resembles the filter pattern.

**Filter types visible:**
- **Edge detectors**: sharp transitions (positive-to-negative or vice versa) — respond to K-complexes and sharp waves
- **Sinusoidal filters**: oscillatory patterns resembling delta, theta, or alpha rhythms
- **Baseline offsets**: detect sustained deflections (e.g., slow-wave upstates)

The diversity of learned filters suggests the model uses a combination of frequency content and transient event detection for staging.

---
## 3. Conv1 Activation Maps Across Stages

Plot the post-ReLU activation of each conv1 filter when processing Wake, N2, N3, and REM epochs. This reveals which filters respond preferentially to each stage.

In [ ]:
stages_to_plot = ['Wake', 'N2', 'N3', 'REM']
fig, axes = plt.subplots(len(stages_to_plot), 16, figsize=(20, 8))
fig.suptitle('Conv1 Activation Maps (post-ReLU, before pooling)', fontsize=14, y=1.01)

t = np.arange(3000) / sfreq

for row, stage in enumerate(stages_to_plot):
    acts = stage_outputs[stage]['conv1'].squeeze(0).cpu().numpy()  # (16, 3000)
    axes[row, 0].set_ylabel(stage, fontsize=9, rotation=0, ha='right', va='center')
    for col in range(16):
        ax = axes[row, col]
        ax.plot(t, acts[col], linewidth=0.3, color='black')
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(f'F{col+1}', fontsize=7)
        if row == len(stages_to_plot) - 1:
            ax.set_xticks([0, 15, 30])
            ax.tick_params(labelsize=6)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'conv1_activations.png'), dpi=150, bbox_inches='tight')
plt.show()

### Activation Map Observations

- **N3** often shows strong, rhythmic activations in multiple filters — reflecting the high-amplitude synchronized slow-wave activity that dominates deep sleep.
- **Wake** tends to activate filters detecting higher-frequency content (alpha/beta), with more irregular activation patterns.
- **N2** may show intermediate activation levels with occasional strong bursts corresponding to sleep spindles or K-complexes.
- **REM** typically shows lower overall activation across most filters, consistent with its low-amplitude, desynchronized EEG.

Different filters appear to specialize: some respond strongly only to N3 (delta detectors), while others are broadly activated across stages.

---
## 4. Mean Activation per Filter by Stage

Quantify which conv1 filters are most selective for each sleep stage by comparing mean activation strengths.

In [ ]:
all_stage_acts = {s: [] for s in STAGE_NAMES}
for i, s in enumerate(STAGE_NAMES):
    idxs = np.where(y == i)[0][:50]
    for idx in idxs:
        t = torch.tensor(X[idx], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        _ = model.conv1(t)
        all_stage_acts[s].append(activations['conv1'].squeeze(0).cpu().numpy())
    all_stage_acts[s] = np.mean(all_stage_acts[s], axis=0)  # (16, 3000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean activation amplitude per filter
ax = axes[0]
x = np.arange(16)
for i, s in enumerate(STAGE_NAMES):
    mean_amps = np.mean(np.abs(all_stage_acts[s]), axis=1)
    ax.plot(x, mean_amps, 'o-', color=STAGE_COLORS[i], label=s, linewidth=1.2, markersize=4)
ax.set_xlabel('Filter Index')
ax.set_ylabel('Mean |Activation|')
ax.set_title('Mean Conv1 Activation per Filter by Stage')
ax.set_xticks(x)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# Which filter is most activated per stage
ax = axes[1]
preferred = {}
for s in STAGE_NAMES:
    amps = np.mean(np.abs(all_stage_acts[s]), axis=1)
    preferred[s] = np.argmax(amps)
    ax.bar(s, amps[preferred[s]], color='#3366cc', alpha=0.8)
    ax.text(s, amps[preferred[s]] + 0.01, f'F{preferred[s]+1}',
            ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Max Mean |Activation|')
ax.set_title('Most Activated Conv1 Filter per Stage')
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'filter_selectivity.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Preferred filter per stage:')
for s, f in preferred.items():
    print(f'  {s}: Filter {f+1}')

---
## 5. Deep Layers: Conv2 and Conv3 Feature Maps

Later layers build on conv1 outputs to detect more complex temporal patterns. We visualize the mean activation per channel for conv2 and conv3.

In [ ]:
for layer_name in ['conv2', 'conv3']:
    n_ch = stage_outputs['Wake'][layer_name].shape[1]
    fig, axes = plt.subplots(len(stages_to_plot), 1, figsize=(14, 8))
    fig.suptitle(f'{layer_name} — Mean Activation per Channel by Stage', fontsize=13)
    
    for row, stage in enumerate(stages_to_plot):
        acts = stage_outputs[stage][layer_name].squeeze(0).cpu().numpy()
        mean_per_ch = np.mean(np.abs(acts), axis=1)
        axes[row].bar(range(n_ch), mean_per_ch, color=STAGE_COLORS[STAGE_NAMES.index(stage)],
                      alpha=0.7, width=0.9)
        axes[row].set_ylabel(f'{stage}\nMean |Act|', fontsize=9)
        axes[row].set_xlim(-1, n_ch)
        axes[row].grid(True, alpha=0.15, axis='y')
        if row < len(stages_to_plot) - 1:
            axes[row].set_xticks([])
    
    axes[-1].set_xlabel('Channel Index')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{layer_name}_activations.png'), dpi=150, bbox_inches='tight')
    plt.show()

---
## 6. Raw EEG vs. Filter Response Comparison

Side-by-side view of the raw EEG epoch and the corresponding conv1 activation pattern for a select filter that differentiates stages.

In [ ]:
# Find the filter with highest variance across stages
mean_amps_per_stage = []
for s in STAGE_NAMES:
    mean_amps_per_stage.append(np.mean(np.abs(all_stage_acts[s]), axis=1))
mean_amps_per_stage = np.array(mean_amps_per_stage)  # (5, 16)
var_across_stages = np.var(mean_amps_per_stage, axis=0)
best_filter = int(np.argmax(var_across_stages))
print(f'Most stage-discriminative filter: conv1 Filter {best_filter+1} '
      f'(variance across stages = {var_across_stages[best_filter]:.4f})')

t = np.arange(3000) / sfreq
fig, axes = plt.subplots(4, 2, figsize=(14, 10))
fig.suptitle(f'Raw EEG vs Conv1 Filter {best_filter+1} Response', fontsize=13)

for row, stage in enumerate(stages_to_plot):
    epoch = stage_examples[stage]
    act = stage_outputs[stage]['conv1'].squeeze(0).cpu().numpy()[best_filter]
    
    axes[row, 0].plot(t, epoch, color='black', linewidth=0.5)
    axes[row, 0].set_ylabel(stage, fontsize=9)
    axes[row, 0].set_ylim(-4, 4)
    if row == 0:
        axes[row, 0].set_title('Raw EEG (normalized)')
    
    axes[row, 1].plot(t, act, color=STAGE_COLORS[STAGE_NAMES.index(stage)], linewidth=0.6)
    axes[row, 1].fill_between(t, act, alpha=0.15,
                              color=STAGE_COLORS[STAGE_NAMES.index(stage)])
    axes[row, 1].set_ylim(0, max(act.max(), 0.1))
    axes[row, 1].set_ylabel('Activation', fontsize=9)
    if row == 0:
        axes[row, 1].set_title(f'Conv1 Filter {best_filter+1} Response')
    
    if row == 3:
        axes[row, 0].set_xlabel('Time (s)')
        axes[row, 1].set_xlabel('Time (s)')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eeg_vs_filter_response.png'), dpi=150, bbox_inches='tight')
plt.show()

### Raw vs. Filter Response Analysis

The most stage-discriminative filter highlights how the model separates stages:

- **Wake** often shows low, sparse activation in delta-tuned filters but stronger activation in high-frequency filters.
- **N2** activation pattern shows occasional sharp bursts corresponding to K-complexes and sleep spindles.
- **N3** produces strong, rhythmic activation in low-frequency filters, tracking the slow-wave oscillation cycle.
- **REM** typically shows low and relatively flat activation across most filters, similar to Wake but without the alpha spindles.

The convolution operation essentially decomposes the EEG into frequency-specific envelopes. The fact that different filters specialize by stage confirms the model has learned physiologically meaningful features.

---
## 7. Frequency Response of Learned Filters

Compute the frequency response of each conv1 filter to interpret which frequency bands they detect.

In [ ]:
from scipy import signal as sp_signal

freq_response = np.zeros((16, 512))
for i in range(16):
    w, h = sp_signal.freqz(conv1_weights[i, 0, ::-1], a=[1], worN=512)
    freq_response[i] = np.abs(h)

freqs_hz = w * sfreq / (2 * np.pi)

BAND_RANGES = {'Delta': (0.5, 4), 'Theta': (4, 8), 'Alpha': (8, 13), 'Beta': (13, 30)}
BAND_COLORS = {'Delta': '#1f77b4', 'Theta': '#ff7f0e', 'Alpha': '#2ca02c', 'Beta': '#d62728'}

fig, axes = plt.subplots(4, 4, figsize=(14, 10))
fig.suptitle('Frequency Response of Conv1 Filters', fontsize=13)

for i in range(16):
    ax = axes[i // 4, i % 4]
    ax.plot(freqs_hz, freq_response[i] / freq_response[i].max(), color='black', linewidth=1)
    ax.set_xlim(0, 30)
    ax.set_title(f'Filter {i+1}', fontsize=9)
    ax.set_ylim(0, 1.1)
    
    for bn, (lo, hi) in BAND_RANGES.items():
        ax.axvspan(lo, hi, alpha=0.06, color=BAND_COLORS[bn])
    
    if i >= 12:
        ax.set_xlabel('Hz', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'filter_frequency_response.png'), dpi=150, bbox_inches='tight')
plt.show()

# Classify filters by dominant frequency band
print('Filter frequency band preferences:')
for i in range(16):
    spec = freq_response[i] / freq_response[i].max()
    band_power = {}
    for bn, (lo, hi) in BAND_RANGES.items():
        mask = (freqs_hz >= lo) & (freqs_hz <= hi)
        band_power[bn] = np.trapz(spec[mask], freqs_hz[mask])
    dominant = max(band_power, key=band_power.get)
    print(f'  Filter {i+1:2d}: {dominant:>5s}  ({band_power["Delta"]:.2f}D '
          f'{band_power["Theta"]:.2f}T {band_power["Alpha"]:.2f}A {band_power["Beta"]:.2f}B)')

### Frequency Response Interpretation

The conv1 filters span the EEG frequency spectrum from delta through beta. This spectral diversity is crucial for sleep staging because:

- **Delta-preferring filters** strongly activate during N3, enabling deep-sleep detection
- **Alpha-preferring filters** help identify Wake (eyes-closed alpha rhythm)
- **Beta-preferring filters** contribute to Wake and REM discrimination
- Filters with broad/mixed responses likely detect transient events like spindles (sigma ~12–15 Hz)

---
## Summary of Findings

### What the CNN learns:

1. **Layer 1 (Conv1)**: 16 primitive temporal pattern detectors — edge detectors, oscillatory filters tuned to delta/theta/alpha/beta bands, and baseline shift detectors. These function as learned frequency-selective channels.

2. **Layer 2 (Conv2)**: Combines conv1 outputs into 32 more complex features, likely encoding combinations of frequency bands (e.g., "delta + theta" or "spindle + K-complex").

3. **Layer 3 (Conv3)**: Further integrates features, likely encoding whole-epoch macro patterns (e.g., "sustained high delta for 30s" → N3).

### Temporal features important for classification:

- **Slow-wave amplitude**: Strong, rhythmic low-frequency (0.5–4 Hz) power indicates N3
- **Alpha rhythm presence**: Sustained ~10 Hz activity indicates Wake
- **Spindle events**: Short 12–15 Hz bursts during N2
- **Overall desynchronization**: Low amplitude across all bands (REM, N1, Wake)
- **Signal variability**: N3 shows low-frequency, high-variance patterns; Wake shows high-frequency, moderate-variance patterns

### Limitations of this analysis:
- Single subject only
- Single EEG channel limits the feature space
- Small kernel sizes (7 samples = 70ms) restrict low-frequency resolution in early layers